# Tarea 2 · Parte 2 — Evaluación y extracción estructurada

**Pregunta guía:** ¿Pueden convertirse documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

Este notebook mide qué tan bien leen los modelos (CER en documentos, WER en audio), los compara sobre **las mismas muestras**, extrae campos estructurados con un LLM y audita esa extracción a mano.

| | |
|---|---|
| **Fecha** | 27 de octubre de 2026 |
| **Datos** | `datos/documentos.csv` (25 documentos) y `datos/transcripciones.csv` (20 clips, 32.3 min) de la Parte 1 |
| **Referencias** | Transcripción manual propia de 10 documentos y 10 clips (`datos/raw/referencias/`, no versionada: puede tener nombres reales) |

## 1. Muestra de evaluación y configuraciones (fijadas antes de ver métricas)

Esta sección se escribió y se subió a GitHub **antes** de terminar las transcripciones de referencia y antes de calcular cualquier CER o WER. Ninguna configuración se cambia después según los resultados.

### Muestra

- **Documentos (10):** 3 impresos, 3 fotos y 4 manuscritos. Los manuscritos tienen un elemento más porque son el caso difícil.
- **Audios (10):** 7 fuentes o autores distintos y 5 condiciones: calle, fiesta con ruido, marcha con ruido, testimonio y lectura en estudio. Incluye clips con 2 o más hablantes.
- **Criterio:** dentro de cada tipo se eligieron los elementos de **menor costo de transcripción manual**. Así quedaron fuera `doc_003`, una página casi vacía, y las fotos de más de 5 000 caracteres (`doc_021`–`doc_025`). En audio se excluyeron la conferencia (`clip_003`) y los testimonios de 3–4 min (`clip_014`, `clip_015`), y por la misma razón se tomaron 2 de las 6 lecturas de OpenSLR.
- **Sesgo conocido:** el criterio favorece páginas y audios cortos. Las métricas describen esta muestra, no necesariamente a las páginas de periódico más densas.

### Transcripción de referencia
Se hace a mano y sin mirar la salida de ningún modelo. Se copia línea por línea y con la ortografía original. Las palabras ilegibles se marcan `[?]` y los tramos que no se entienden, `[inaudible]`. En audio se omiten las muletillas (`eh`, `mmm`) y los números se escriben con cifras. La guía completa está en `datos/raw/referencias/LEEME.md`.

### Normalización (la misma para referencia e hipótesis)
Se usa `metricas.normalizar`:
- quita las marcas `[HABLANTE_n]`, `[?]` y `[inaudible]`;
- convierte los números a palabras (`19` → `diecinueve`);
- pasa todo a minúsculas, quita la puntuación y deja espacios simples;
- **quita las tildes y diacríticos** (`á` → `a`, `ñ` → `n`).

CER y WER se calculan **sobre los textos originales**, sin seudonimizar, para que el reemplazo de nombres no cuente como error. El valor global agrega los errores de los 10 casos (jiwer) y también se reporta el valor de cada caso.

> **Cambio antes de medir (27/10/2026):** el plan inicial conservaba las tildes y no convertía números. Al revisar las primeras referencias manuales se vio que las tildes no estaban marcadas de forma pareja y que algunos números estaban en palabras y otros en cifras. Las dos cosas son diferencias de tecleo, no errores de lectura del modelo, así que la normalización las iguala. El cambio se hizo **antes de calcular cualquier CER o WER**.

### Modelos

| Modalidad | Rol | Configuración |
|---|---|---|
| OCR | **Línea base** | PaddleOCR 3.0.3 por defecto: PP-OCRv5 *mobile* det + rec, orientación de página, desdoblado UVDoc, orientación de línea y orden de lectura de Paddle. Solo se cambian los modelos *server* por los *mobile*, porque los *server* no corren en este CPU (Parte 1). |
| OCR | Ajustado | PaddleOCR de la Parte 1: *mobile*, detección a 1600 px, sin preprocesado de página y orden de lectura por columnas. Se definió en la Parte 1, antes de que existieran las referencias. |
| OCR | Alternativo | Tesseract 5.5.1 (tesserocr 2.9.1, `spa` de tessdata 4.1.0, LSTM, PSM 3) |
| ASR | **Línea base** | faster-whisper `small` (int8, CPU, `language=es`, VAD Silero, beam 5) |
| ASR | Mejorado | faster-whisper `large-v3-turbo`, con la misma configuración (el modelo de la Parte 1) |
| ASR | Alternativo | Qwen3-ASR 0.6B (transformers, float32, CPU, `language=es`, tramos VAD ≤ 12 s, decodificación greedy) |

La latencia de ASR se mide solo durante la transcripción, sin contar la carga del modelo ni la diarización. Los modelos se ejecutan uno por uno en el mismo equipo (Intel i5 de 2 núcleos). **Factor de tiempo real = segundos de proceso / duración del audio.**

### Extracción con LLM
- **Modelo:** Gemini 3.8 Flash (`gemini-3.8-flash`, API gratuita), temperatura 0 y salida JSON con esquema fijo.
- **Entrada:** los textos **ya seudonimizados**, para no enviar nombres reales a un servicio externo.
- **Campos comunes:** `modalidad`, `tema` y `resumen`.
- **Campos de documentos:** `emisor`, `fecha`, `lugar` y `monto`.
- **Campos de audio:** `numero_hablantes` y `citas_clave`.
- Un campo que no aparece en el texto debe quedar en `null`.

La auditoría humana revisa 20 extracciones sorteadas con semilla fija (10 documentos y 10 audios).

In [1]:
import pandas as pd

docs = pd.read_csv("datos/documentos.csv")
clips = pd.read_csv("datos/transcripciones.csv")
fuentes = pd.read_csv("datos/fuentes.csv").set_index("id")

MUESTRA_DOCS = ["doc_001", "doc_002", "doc_004", "doc_018", "doc_019", "doc_020", "doc_009", "doc_015", "doc_016", "doc_017"]
MUESTRA_CLIPS = ["clip_001", "clip_010", "clip_011", "clip_005", "clip_006", "clip_007", "clip_008", "clip_013", "clip_016", "clip_017"]

m_docs = docs.set_index("id").loc[MUESTRA_DOCS, ["tipo"]].assign(caracteres_ocr=docs.set_index("id")["texto_ocr"].str.len())
m_clips = clips.set_index("id").loc[MUESTRA_CLIPS, ["duracion_seg", "n_hablantes"]].join(fuentes[["tipo", "autor"]])
print("Documentos por tipo:", m_docs["tipo"].value_counts().to_dict())
print(f"Audios: {len(m_clips)} clips, {m_clips['duracion_seg'].sum() / 60:.1f} min, {m_clips['autor'].nunique()} fuentes, condiciones:", sorted(m_clips["tipo"].unique()))
display(m_docs, m_clips)

Documentos por tipo: {'manuscrito': 4, 'impreso': 3, 'foto': 3}
Audios: 10 clips, 11.5 min, 7 fuentes, condiciones: ['calle', 'fiesta, ruido', 'lectura, estudio', 'marcha, ruido', 'testimonio']


,tipo,caracteres_ocr
id,,
doc_001,impreso,362
doc_002,impreso,1038
doc_004,impreso,1516
doc_018,foto,2526
doc_019,foto,2388
doc_020,foto,504
doc_009,manuscrito,1892
doc_015,manuscrito,1861
doc_016,manuscrito,1144


,duracion_seg,n_hablantes,tipo,autor
id,,,,
clip_001,136.3,2,calle,JorgeFlores-pe
clip_010,96.1,1,calle,Gianpier Yair Venegas Ambia
clip_011,77.4,2,calle,Gianpier Yair Venegas Ambia
clip_005,81.6,2,"fiesta, ruido",Rashelley Perez
clip_006,77.4,2,"marcha, ruido",Hellloff233
clip_007,95.0,5,"marcha, ruido",Hellloff233
clip_008,54.3,3,"marcha, ruido",Joaquin Canales
clip_013,53.1,1,testimonio,Chantell Acosta
clip_016,10.2,1,"lectura, estudio","Google (Guevara-Rukoz et al., 2020)"


## 2. Referencias manuales

Las transcripciones manuales están en `datos/raw/referencias/<id>.txt`, que no se versiona. Las métricas se calculan con ese texto original. Para publicarlo en la columna `texto_referencia` de los CSV se aplica la misma seudonimización revisada de la Parte 1: los audios y los manuscritos se seudonimizan; los impresos y las fotos de periódicos públicos, no.

**Control de calidad.** Los dos clips de OpenSLR tienen una transcripción humana publicada por el propio corpus, y las referencias manuales coinciden palabra por palabra con ella (con la normalización de la sección 1). En los demás audios no hay una transcripción externa con qué comparar.

In [2]:
import json
import re
import subprocess
import sys
import tempfile
from pathlib import Path

import jiwer
import plotly.express as px

import metricas
import voz

REF = Path("datos/raw/referencias")
referencia = {p.stem: p.read_text(encoding="utf-8") for p in sorted(REF.glob("*.txt")) if p.read_text(encoding="utf-8").strip()}
pendientes = [i for i in MUESTRA_DOCS + MUESTRA_CLIPS if i not in referencia]
print(f"Referencias manuales listas: {len(referencia)} de 20 · pendientes: {pendientes or 'ninguna'}")

atrib = pd.read_csv("datos/raw/audio/atribucion_audio.csv").set_index("id")
for c in ["clip_016", "clip_017"]:
    print(f"{c} vs transcripción oficial de OpenSLR: WER {metricas.wer([atrib.loc[c, 'transcripcion_openslr']], [referencia[c]]):.3f}")

reemplazos = pd.read_csv("datos/raw/seudonimizacion/reemplazos.csv")
mapa = dict(zip(reemplazos.loc[reemplazos["aplicar"] == "si", "nombre"], reemplazos.loc[reemplazos["aplicar"] == "si", "seudonimo"]))
seudonimizar = lambda t: voz.seudonimizar(t, mapa) if isinstance(t, str) and t else t

# texto_referencia publicado: seudonimizado en audios y manuscritos
clips["texto_referencia"] = clips["id"].map(lambda i: seudonimizar(referencia.get(i, "")))
privado = docs["tipo"] == "manuscrito"
docs["texto_referencia"] = [seudonimizar(referencia.get(i, "")) if p else referencia.get(i, "") for i, p in zip(docs["id"], privado)]
clips.to_csv("datos/transcripciones.csv", index=False)
docs.to_csv("datos/documentos.csv", index=False)
print("texto_referencia completado en", (clips["texto_referencia"] != "").sum(), "clips y", (docs["texto_referencia"] != "").sum(), "documentos")

Referencias manuales listas: 11 de 20 · pendientes: ['doc_001', 'doc_002', 'doc_004', 'doc_018', 'doc_019', 'doc_020', 'doc_009', 'doc_015', 'doc_016']
clip_016 vs transcripción oficial de OpenSLR: WER 0.000
clip_017 vs transcripción oficial de OpenSLR: WER 0.000
texto_referencia completado en 10 clips y 1 documentos


## 3. Audio: línea base y modelos alternativos (WER y latencia)

Los 3 modelos transcribieron los mismos 10 clips con la configuración de la sección 1 (`evaluar_asr.py`, un proceso por modelo, uno tras otro). El WER global suma los errores de los 10 clips y los divide por el total de palabras de las referencias, así que los clips largos pesan más. El factor de tiempo real (RTF) global es el tiempo total de proceso dividido por la duración total del audio.

In [3]:
MODELOS_ASR = {"faster-whisper-small": "Whisper small (línea base)", "faster-whisper": "Whisper large-v3-turbo",
               "qwen3-asr-0.6b": "Qwen3-ASR 0.6B"}
EVAL = Path("datos/raw/audio/eval")
filas = []
for clave, nombre in MODELOS_ASR.items():
    for c in MUESTRA_CLIPS:
        d = json.loads((EVAL / clave / f"{c}.json").read_text(encoding="utf-8"))
        o = jiwer.process_words(metricas.normalizar(referencia[c]), metricas.normalizar(d["texto"]) or "∅")
        filas.append({"modelo": nombre, "id": c, "condicion": fuentes.loc[c, "tipo"], "duracion_seg": d["duracion_seg"],
                      "segundos": d["segundos"], "palabras_ref": o.hits + o.substitutions + o.deletions,
                      "sustituidas": o.substitutions, "borradas": o.deletions, "insertadas": o.insertions, "wer": o.wer,
                      "texto": d["texto"], "segmentos": d["segmentos"]})
asr = pd.DataFrame(filas)
asr["errores"] = asr[["sustituidas", "borradas", "insertadas"]].sum(axis=1)

def resumir(g):
    return pd.Series({"WER": g["errores"].sum() / g["palabras_ref"].sum(), "RTF": g["segundos"].sum() / g["duracion_seg"].sum(),
                      "sustituidas": g["sustituidas"].sum(), "borradas": g["borradas"].sum(), "insertadas": g["insertadas"].sum()})

tabla_asr = asr.groupby("modelo", sort=False).apply(resumir, include_groups=False)
display(tabla_asr.round(3))
por_condicion = asr.groupby(["condicion", "modelo"], sort=False).apply(resumir, include_groups=False)["WER"].unstack()
display(por_condicion.round(3))
display(asr.pivot(index=["id", "condicion"], columns="modelo", values="wer").round(3))

,WER,RTF,sustituidas,borradas,insertadas
modelo,,,,,
Whisper small (línea base),0.320,0.531,174.0,65.0,197.0
Whisper large-v3-turbo,0.302,1.822,122.0,53.0,237.0
Qwen3-ASR 0.6B,0.379,0.949,206.0,63.0,248.0


modelo,Whisper small (línea base),Whisper large-v3-turbo,Qwen3-ASR 0.6B
condicion,,,
calle,0.264,0.233,0.343
"fiesta, ruido",0.441,0.419,0.520
"marcha, ruido",0.400,0.410,0.437
testimonio,0.183,0.139,0.209
"lectura, estudio",0.037,0.037,0.037


,modelo,Qwen3-ASR 0.6B,Whisper large-v3-turbo,Whisper small (línea base)
id,condicion,,,
clip_001,calle,0.387,0.281,0.293
clip_005,"fiesta, ruido",0.520,0.419,0.441
clip_006,"marcha, ruido",0.265,0.193,0.283
clip_007,"marcha, ruido",0.664,0.671,0.503
clip_008,"marcha, ruido",0.406,0.398,0.438
clip_010,calle,0.356,0.207,0.264
clip_011,calle,0.267,0.188,0.222
clip_013,testimonio,0.209,0.139,0.183
clip_016,"lectura, estudio",0.067,0.067,0.067


In [4]:
fig = px.bar(por_condicion.reset_index().melt(id_vars="condicion", var_name="modelo", value_name="WER"),
             x="condicion", y="WER", color="modelo", barmode="group", text_auto=".2f",
             title="El ruido de marchas y fiestas multiplica el error: el WER sube de 0.04 en estudio a 0.40–0.52",
             labels={"condicion": "Condición de grabación"})
fig.show()

fig = px.scatter(tabla_asr.reset_index(), x="RTF", y="WER", text="modelo", size=[1, 1, 1],
                 title="Whisper turbo apenas mejora a small (0.30 vs 0.32) pero tarda 3,4 veces más",
                 labels={"RTF": "Factor de tiempo real (segundos de proceso / segundo de audio)"})
fig.update_traces(textposition="top center")
fig.show()

### Alucinaciones en silencio y referencias incompletas

En los 3 modelos predominan las palabras **insertadas**. Puede haber dos causas:

1. **Alucinaciones.** En los últimos ~20 s de `clip_007`, que son casi silencio, Whisper turbo escribe la misma palabra decenas de veces, aunque el VAD estaba activado.
2. **Referencias incompletas.** En varios tramos los 3 modelos coinciden en palabras que no están en la referencia manual, sobre todo preguntas del entrevistador, voces de fondo y repeticiones. Se marcaron esos tramos para volver a escucharlos, pero se decidió no modificar las referencias.

Esta segunda causa es una **limitación**: si a la referencia le faltan palabras dichas, las tres métricas suben por igual. Por eso el WER absoluto puede estar sobreestimado, aunque la comparación entre modelos sigue siendo válida porque todos se miden contra el mismo texto.

In [5]:
def repeticiones(segmentos, minimo=3):
    # segmentos idénticos seguidos (la firma típica de una alucinación de Whisper)
    textos = [metricas.normalizar(s[2]) for s in segmentos]
    rachas, n = [], 1
    for a, b in zip(textos, textos[1:] + [None]):
        if a == b:
            n += 1
        else:
            if n >= minimo:
                rachas.append((a, n))
            n = 1
    return rachas

for (modelo, c), segs in asr.set_index(["modelo", "id"])["segmentos"].items():
    for texto, n in repeticiones(segs):
        print(f"{modelo} · {c}: '{texto}' repetido {n} veces seguidas")

turbo7 = asr[(asr["modelo"] == "Whisper large-v3-turbo") & (asr["id"] == "clip_007")].iloc[0]
sin_cola = " ".join(t for ini, fin, t in turbo7["segmentos"] if ini < 93.5)
print(f"Whisper turbo en clip_007: WER {turbo7['wer']:.3f} → {metricas.wer([referencia['clip_007']], [sin_cola]):.3f} sin la cola alucinada")
print("Palabras insertadas / errores totales:", (asr.groupby("modelo", sort=False)["insertadas"].sum() / asr.groupby("modelo", sort=False)["errores"].sum()).round(2).to_dict())

Whisper large-v3-turbo · clip_007: 'listo' repetido 23 veces seguidas
Whisper turbo en clip_007: WER 0.671 → 0.510 sin la cola alucinada
Palabras insertadas / errores totales: {'Whisper small (línea base)': 0.45, 'Whisper large-v3-turbo': 0.58, 'Qwen3-ASR 0.6B': 0.48}


In [6]:
# resultados por clip para el dashboard: transcripciones seudonimizadas, sin nombres reales
pub = asr.assign(texto=asr["texto"].map(seudonimizar), rtf=asr["segundos"] / asr["duracion_seg"])
pub[["id", "modelo", "condicion", "duracion_seg", "segundos", "rtf", "wer", "sustituidas", "borradas", "insertadas", "texto"]].to_csv(
    "datos/evaluacion_asr.csv", index=False)
print("datos/evaluacion_asr.csv:", pub.shape)

# verificación: ningún nombre revisado con aplicar=si queda en lo publicado
publicado = " ".join(pd.read_csv(p).astype(str).agg(" ".join, axis=1).str.cat(sep=" ")
                     for p in ["datos/evaluacion_asr.csv", "datos/transcripciones.csv"])
publicado += " " + " ".join(docs.loc[privado, "texto_referencia"].fillna(""))
quedan = [n for n in mapa if re.search(rf"(?<!\\w){re.escape(n)}(?!\\w)", publicado)]
print("Nombres reales en textos publicados de audio y manuscritos:", len(quedan))

datos/evaluacion_asr.csv: (30, 14)
Nombres reales en textos publicados de audio y manuscritos: 0
